# Lecture 4 — Regularization, Cross-Validation, and Hyperparameter Selection

Lecture 3 optimized $\theta$ for a fixed $\alpha$. Here we add the next layer: choose $\alpha$ using cross-validation.

We will keep the test set untouched until the final evaluation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42


## 1. Load and inspect the data

The breast-cancer dataset contains 30 numeric attributes. We use the full feature vector for the classifier; plots later use only two features for visualization.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'classes: {data.target_names}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


## 2. Why does alpha matter?

Our conceptual objective is

$J(\theta;\alpha)=L(\theta)+\alpha\|\theta\|_2^2$.

Small $\alpha$ emphasizes fitting the training data. Increasing $\alpha$ penalizes large parameter vectors more strongly. The best value is not chosen from training accuracy; it is selected using validation performance.

In [ ]:
alphas = np.r_[1e-5, np.arange(0.05, 1.01, 0.05)]

def make_model(alpha):
    # sklearn's SGDClassifier uses alpha as the regularization strength.
    return SGDClassifier(
        loss='hinge', penalty='l2', alpha=float(alpha),
        max_iter=5000, tol=1e-4, random_state=RANDOM_STATE
    )


## 3. Implement 5-fold cross-validation

For a fixed alpha, every observation is used for validation exactly once. We train on the other four folds and average the five validation accuracies.

In [ ]:
def cross_validate_alpha(X, y, alpha, n_splits=5, random_state=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
        model = make_model(alpha)
        model.fit(X[train_idx], y[train_idx])
        score = accuracy_score(y[val_idx], model.predict(X[val_idx]))
        fold_scores.append(score)
        print(f'alpha={alpha:.5g} | fold {fold}: validation accuracy={score:.4f}')

    return np.array(fold_scores)


## 4. Search over alpha

Now cross-validation becomes a hyperparameter-search strategy. We calculate the mean validation score for every candidate alpha and select

$\alpha^*=\arg\max_\alpha S(\alpha)$.


In [ ]:
cv_means = []
cv_stds = []
all_fold_scores = {}

for alpha in alphas:
    scores = cross_validate_alpha(X_train, y_train, alpha, n_splits=5, random_state=RANDOM_STATE)
    all_fold_scores[float(alpha)] = scores
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f'  mean={scores.mean():.4f}, std={scores.std():.4f}\n')

cv_means = np.array(cv_means)
cv_stds = np.array(cv_stds)
best_index = int(np.argmax(cv_means))
alpha_star = float(alphas[best_index])
print(f'alpha* = {alpha_star:.5g}')
print(f'best mean CV accuracy = {cv_means[best_index]:.4f}')


In [ ]:
plt.figure(figsize=(9, 5))
plt.errorbar(alphas, cv_means, yerr=cv_stds, marker='o', capsize=3)
plt.axvline(alpha_star, linestyle='--', label=f'alpha* = {alpha_star:.5g}')
plt.xlabel('Regularization strength alpha')
plt.ylabel('5-fold validation accuracy')
plt.title('Cross-validation accuracy vs. alpha')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## 5. Retrain the final model

After selecting alpha*, we train one final model on **all** available training data. The test set has not been used during model selection.

In [ ]:
final_model = make_model(alpha_star)
final_model.fit(X_train, y_train)

train_accuracy = accuracy_score(y_train, final_model.predict(X_train))
test_accuracy = accuracy_score(y_test, final_model.predict(X_test))

print(f'alpha*              : {alpha_star:.5g}')
print(f'training accuracy   : {train_accuracy:.4f}')
print(f'final test accuracy : {test_accuracy:.4f}')


## 6. Visualize the decision boundary

The model uses all 30 standardized features. For a 2-D visualization only, we train a separate two-feature model on the first two standardized features. This plot is therefore a visualization of the idea, not the final 30-dimensional classifier.

In [ ]:
X2 = X_train[:, :2]
two_d_model = make_model(alpha_star)
two_d_model.fit(X2, y_train)

x_min, x_max = X2[:, 0].min() - 1, X2[:, 0].max() + 1
y_min, y_max = X2[:, 1].min() - 1, X2[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = two_d_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, zz, alpha=0.15)
plt.scatter(X2[:, 0], X2[:, 1], c=y_train, edgecolor='k', alpha=0.7)
plt.xlabel('Standardized feature 1')
plt.ylabel('Standardized feature 2')
plt.title('Two-feature visualization of the linear decision boundary')
plt.show()


## 7. The complete strategy

The important distinction from Lecture 3 is now explicit:

1. **For each alpha:** optimize the model parameters theta.
2. **Cross-validation:** estimate how well that alpha generalizes.
3. **Model selection:** choose alpha*.
4. **Final training:** retrain theta using alpha* and all training data.
5. **Final evaluation:** use the untouched test set once.

This separates parameter optimization from hyperparameter selection.